# NutriTrack API Test
Test script for the NutriTrack FastAPI server.

**Prerequisites**: Start the API server first:
```bash
cd d:\Project\Code\nutritrack-documentation\app
python templates/api.py
```

In [4]:
import requests
import json
import os

BASE_URL = "http://localhost:8000"
DATA_DIR = r"D:\Project\Code\nutritrack-documentation\data\images\food"

## 1. Health Check

In [5]:
res = requests.get(f"{BASE_URL}/health")
print(f"Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

Status: 200
{
  "status": "ok",
  "qwen_model": "qwen.qwen3-vl-235b-a22b",
  "usda_ready": true
}


## 2. Analyze Food Image — fast_food.jpg

In [8]:
img_path = r"D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg"
print(f"Uploading: {img_path}")
print(f"File size: {os.path.getsize(img_path) / 1024:.0f} KB")

with open(img_path, "rb") as f:
    res = requests.post(
        f"{BASE_URL}/analyze",
        files={"file": ("fast_food.jpg", f, "image/jpeg")}
    )

print(f"\nStatus: {res.status_code}")
data = res.json()
print(f"Message: {data.get('message')}")
print(f"Dishes found: {len(data.get('data', []))}")

Uploading: D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg
File size: 93 KB

Status: 200
Message: Phát hiện 1 món ăn.
Dishes found: 1


In [9]:
# Pretty print full results
for i, dish in enumerate(data["data"], 1):
    t = dish["total_nutrition"]
    print(f"\n{'='*60}")
    print(f"🍽️  {i}. {dish['name']} ({dish['vi_name']})")
    print(f"   AI Calories: {dish['qwen_estimated_calories']} kcal")
    print(f"   USDA Total → Cal: {t['calories']:.0f} | Pro: {t['protein']:.1f}g | Carb: {t['carbs']:.1f}g | Fat: {t['fat']:.1f}g")
    print(f"   Avg Calories: {dish['average_calories']:.0f}")
    print(f"   Ingredients ({len(dish['ingredients'])}):\n")
    for ing in dish["ingredients"]:
        n = ing["nutrition"]
        print(f"     - {ing['name']:20s} {ing['weight_g']:6.0f}g  Cal:{n['calories']:5.0f}  P:{n['protein']:4.1f}  C:{n['carbs']:4.1f}  F:{n['fat']:4.1f}")


🍽️  1. Grilled Pork Chop Rice (Cơm Tấm Sườn Nướng)
   AI Calories: 750.0 kcal
   USDA Total → Cal: 1029 | Pro: 63.6g | Carb: 112.7g | Fat: 33.6g
   Avg Calories: 890
   Ingredients (12):

     - Grilled pork chop       200g  Cal:  380  P:36.7  C:15.6  F:18.1
     - Steamed rice            180g  Cal:  272  P: 5.8  C:61.0  F: 0.5
     - Fried egg                60g  Cal:  118  P: 8.2  C: 0.5  F: 8.9
     - Pickled vegetables (carrot and daikon)     50g  Cal:   23  P: 0.5  C: 5.1  F: 0.1
     - Pork meatloaf (chả lụa)     70g  Cal:  105  P: 7.0  C:14.0  F: 3.5
     - Tomato slices            40g  Cal:   36  P: 1.7  C: 5.1  F: 1.0
     - Cucumber slices          30g  Cal:   10  P: 0.1  C: 2.0  F: 0.0
     - Fried shallots           10g  Cal:   30  P: 0.2  C: 1.7  F: 0.0
     - Scallion oil              5g  Cal:    2  P: 0.1  C: 0.4  F: 0.0
     - Fresh lettuce            20g  Cal:    4  P: 0.2  C: 0.7  F: 0.0
     - Chili pepper              5g  Cal:    5  P: 0.0  C: 0.5  F: 0.0
     - Fi

## 3. Analyze Food Image — com_tam.jpg

In [ ]:
img_path = os.path.join(DATA_DIR, "com_tam.jpg")
print(f"Uploading: {img_path}")

with open(img_path, "rb") as f:
    res = requests.post(
        f"{BASE_URL}/analyze",
        files={"file": ("com_tam.jpg", f, "image/jpeg")}
    )

data = res.json()
print(f"Status: {res.status_code} | {data.get('message')}")

for i, dish in enumerate(data["data"], 1):
    t = dish["total_nutrition"]
    print(f"\n🍽️  {i}. {dish['name']} ({dish['vi_name']})")
    print(f"   Cal: {t['calories']:.0f} | Pro: {t['protein']:.1f}g | Carb: {t['carbs']:.1f}g | Fat: {t['fat']:.1f}g")
    for ing in dish["ingredients"]:
        print(f"     - {ing['name']:20s} {ing['weight_g']:5.0f}g")

## 4. Error Handling — Invalid File

In [ ]:
# Test with non-image file
res = requests.post(
    f"{BASE_URL}/analyze",
    files={"file": ("test.txt", b"not an image", "text/plain")}
)
print(f"Status: {res.status_code}")
print(f"Response: {res.json()}")
assert res.status_code == 400, "Should reject non-image files"
print("✅ Error handling works correctly!")